# Full Evaluation — BiLSTM + Autoencoder + Fusion

In [ ]:
# Cell 1 — Clone and setup
import os, sys
if not os.path.exists('/content/hallucination-detector'):
    !git clone https://github.com/theek-sh-nahh/hallucination-detector.git
else:
    !cd /content/hallucination-detector && git pull origin main
os.chdir('/content/hallucination-detector')
sys.path.insert(0, '/content/hallucination-detector')
print('Ready. src:', os.listdir('src'))

In [ ]:
# Cell 2 — Install deps
!pip install -q sentence-transformers seaborn
print('Done.')

In [ ]:
# Cell 3 — Upload processed_data.zip
from google.colab import files
import zipfile
uploaded = files.upload()
os.makedirs('data/processed', exist_ok=True)
with zipfile.ZipFile('processed_data.zip', 'r') as z:
    z.extractall('data/processed')
print('Data ready:', os.listdir('data/processed'))

In [ ]:
# Cell 4 — Mount Drive and load models
from google.colab import drive
drive.mount('/content/drive')

drive_dir = '/content/drive/MyDrive/hallucination-detector-models'

os.makedirs('models/lstm',        exist_ok=True)
os.makedirs('models/autoencoder', exist_ok=True)

import shutil
shutil.copy(f'{drive_dir}/lstm/lstm_final.pt',           'models/lstm/lstm_final.pt')
shutil.copy(f'{drive_dir}/autoencoder/ae_final.pt',      'models/autoencoder/ae_final.pt')
shutil.copy(f'{drive_dir}/fusion_params.json',           'models/fusion_params.json')
shutil.copy(f'{drive_dir}/ae_threshold.json',            'models/ae_threshold.json')

print('Models loaded from Drive:')
print(os.listdir('models/lstm'))
print(os.listdir('models/autoencoder'))
print(os.listdir('models'))

In [ ]:
# Cell 5 — Run full evaluation
import numpy as np
import torch
from src.evaluate import run_full_evaluation

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

X_test = np.load('data/processed/X_test.npy')
y_test = np.load('data/processed/y_test.npy')
print(f'Test set: {X_test.shape}')

summary, halluc_conf = run_full_evaluation(
    lstm_path      = 'models/lstm/lstm_final.pt',
    ae_path        = 'models/autoencoder/ae_final.pt',
    params_path    = 'models/fusion_params.json',
    threshold_path = 'models/ae_threshold.json',
    X_test         = X_test,
    y_test         = y_test,
    device         = device,
    output_dir     = 'models/evaluation'
)

In [ ]:
# Cell 6 — Save evaluation results to Drive
from google.colab import drive
import shutil, os

eval_drive = f'{drive_dir}/evaluation'
os.makedirs(eval_drive, exist_ok=True)

for f in os.listdir('models/evaluation'):
    shutil.copy(f'models/evaluation/{f}', f'{eval_drive}/{f}')

print('Evaluation results saved to Drive:')
print(os.listdir(eval_drive))